# Оценка OCR после LLM обработки

Анализ нормализованного текста и структуры документа после обработки (из `data/02_normalized_text/`).
Включает breadcrumbs, иерархию разделов, таблицы и изображения.

In [ ]:
import json
import re
from pathlib import Path
from IPython.display import display, Markdown, Image as IPyImage

# Автоопределение: Docker (/workspace) или локально
if Path('/workspace/data').exists():
    BASE = Path('/workspace')
else:
    BASE = Path.cwd().parent.parent  # notebooks/eval/ -> корень проекта

NORM_DIR = BASE / 'data' / '02_normalized_text'
OCR_DIR  = BASE / 'data' / '01_extracted_pages'
print(f'BASE: {BASE}')
print(f'NORM_DIR exists: {NORM_DIR.exists()}')

# Список документов
norm_docs = sorted([d for d in NORM_DIR.iterdir() if d.is_dir()])
print('Документы в 02_normalized_text:')
for d in norm_docs:
    pages = sorted(d.glob('page_*.json'))
    imgs  = sorted(d.glob('page_*_img*.*'))
    print(f'  {d.name}: {len(pages)} страниц, {len(imgs)} картинок')

## Логика построения breadcrumbs

In [ ]:
# Регулярка как в services/indexer/common/chunker.py
HEADER_RE = re.compile(r'^(?:#+\s+)?(\d+(?:\.\d+)*)\s+([А-ЯЁ][А-Яа-яЁё \-,]{3,100})\.?$')

def build_breadcrumbs(text: str) -> list[tuple[int, str, list[str]]]:
    """Возвращает [(start_line_idx, заголовок, breadcrumb_path), ...]
    Идём по строкам, ведём header-стек как в chunker.py."""
    lines = text.split('\n')
    sections = []
    stack: list[tuple[int, str]] = []
    last_idx = 0
    for i, line in enumerate(lines):
        m = HEADER_RE.match(line.strip())
        if m:
            num = m.group(1)
            title = m.group(2).strip()
            level = num.count('.') + 1
            full = f'{num} {title}'
            while stack and stack[-1][0] >= level:
                stack.pop()
            stack.append((level, full))
            path = [h for _, h in stack]
            sections.append((i, full, path))
            last_idx = i
    return sections

def print_toc(sections):
    """Красивый вывод оглавления."""
    for _, header, path in sections:
        indent = '  ' * (len(path) - 1)
        print(f'{indent}└─ {header}')

In [ ]:
# --- Выберите документ ---
NORM_DOC = norm_docs[0]  # или укажите NORM_DIR / 'имя_папки'

# Загружаем все страницы
norm_pages = []
for p in sorted(NORM_DOC.glob('page_*.json')):
    try:
        d = json.loads(p.read_text(encoding='utf-8'))
        norm_pages.append(d)
    except Exception as e:
        print(f'Ошибка {p.name}: {e}')

# Считаем breadcrumbs по полному тексту
full_text = '\n\n'.join((p.get('text') or '').strip() for p in norm_pages)
all_sections = build_breadcrumbs(full_text)

print(f'Документ: {NORM_DOC.name}')
print(f'Страниц: {len(norm_pages)}, текста: {len(full_text)} символов')
print(f'Разделов (заголовков): {len(all_sections)}\n')

print('=== Оглавление ===')
print_toc(all_sections)

## Постраничный просмотр: текст + таблицы + изображения

In [ ]:
# ── Настройки ──────────────────────────────────────────────
N_PAGE_FROM = 1
N_PAGE_TO   = 5
N_SHOW_TEXT = True
N_SHOW_IMGS = True
N_SHOW_TBLS = True
# ───────────────────────────────────────────────────────────

ocr_sub_norm = OCR_DIR / NORM_DOC.name

for p in norm_pages:
    pg = p.get('page')
    if pg is None or not (N_PAGE_FROM <= pg <= N_PAGE_TO):
        continue

    display(Markdown(f'---\n# Страница {pg}'))

    text = (p.get('text') or '').strip()

    # Breadcrumbs для этой страницы
    if N_SHOW_TEXT:
        if text:
            page_sections = build_breadcrumbs(text)
            if page_sections:
                _, _, path = page_sections[0]
                display(Markdown(f'**Раздел:** {" > ".join(path)}'))
            display(Markdown(f'**Текст ({len(text)} символов):**\n\n{text}'))
        else:
            display(Markdown('*Текст пуст*'))

    # Картинки (PyMuPDF extracted)
    if N_SHOW_IMGS:
        imgs = sorted(NORM_DOC.glob(f'page_{pg:03d}_img*.*'))
        if imgs:
            display(Markdown(f'**Картинки ({len(imgs)} шт.):**'))
            for f in imgs:
                display(IPyImage(filename=str(f), width=500))

    # Таблицы
    if N_SHOW_TBLS:
        tbls = p.get('tables') or []
        if tbls:
            display(Markdown(f'**Таблицы ({len(tbls)} шт.):**'))
            for i, t in enumerate(tbls):
                display(Markdown(f'*Таблица {i+1}:*\n\n{t}'))

## Статистика качества

In [ ]:
# Статистика по документу
stats = {
    'total_pages': len(norm_pages),
    'total_text_chars': sum(len(p.get('text', '')) for p in norm_pages),
    'empty_pages': sum(1 for p in norm_pages if not (p.get('text', '') or '').strip()),
    'pages_with_images': sum(1 for p in norm_pages if p.get('images')),
    'total_images': sum(len(p.get('images', [])) for p in norm_pages),
    'total_tables': sum(len(p.get('tables', [])) for p in norm_pages),
    'avg_text_per_page': 0,
}

if stats['total_pages'] > stats['empty_pages']:
    stats['avg_text_per_page'] = stats['total_text_chars'] // (stats['total_pages'] - stats['empty_pages'])

print('=== Статистика документа ===')
print(f'Страниц: {stats["total_pages"]}')
print(f'  - с текстом: {stats["total_pages"] - stats["empty_pages"]}')
print(f'  - пусто: {stats["empty_pages"]}')
print(f'\nТекст:')
print(f'  - всего: {stats["total_text_chars"]} символов')
print(f'  - среднее на страницу: {stats["avg_text_per_page"]} символов')
print(f'\nОбъекты:')
print(f'  - изображений: {stats["total_images"]}')
print(f'  - таблиц: {stats["total_tables"]}')
print(f'  - разделов (заголовков): {len(all_sections)}')